In [ ]:
import yaml
import pandas as pd
import sys
import logging
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

from utils.features import engineer_features_fold
from utils.labels import generate_stress_targets_for_fold

# =============================================================================
# LOGGING SETUP
# =============================================================================

LOG_DIR = Path("../logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_DIR / "training.log", encoding="utf-8")
file_handler.setFormatter(formatter)
file_handler.setLevel(logging.DEBUG)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
stream_handler.setLevel(logging.INFO)

logger.handlers = []
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

BASE_DATA_PATH = "../data/processed/dataset_features_base.csv"
ES_PATH = "../data/processed/ES_D.csv"
CONFIG_PATH = "../config/model_config.yaml"
OUTPUT_DIR = Path("../artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Load config
# ------------------------------------------------------------------

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# ------------------------------------------------------------------
# Load data
# ------------------------------------------------------------------

base_df = pd.read_csv(BASE_DATA_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
es_df   = pd.read_csv(ES_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print("Loaded base data:", base_df.shape)
print("Loaded ES data:", es_df.shape)
print("Horizons:", config["horizons"])
print("Models:", [m for m, v in config["models"].items() if v["enabled"]])

In [ ]:
import numpy as np
import yaml
import logging
import warnings
from pathlib import Path
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.exceptions import UndefinedMetricWarning

import xgboost as xgb

# =============================================================================
# WARNINGS
# =============================================================================

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# LOGGING SETUP
# =============================================================================

LOG_DIR = Path("../logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_DIR / "training.log", encoding="utf-8")
file_handler.setFormatter(formatter)
file_handler.setLevel(logging.DEBUG)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
stream_handler.setLevel(logging.INFO)

logger.handlers = []
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

# =============================================================================
# MODEL FACTORY
# =============================================================================

def build_model(name, params):
    if name == "logistic_regression":
        return LogisticRegression(**params)
    if name == "xgboost":
        return xgb.XGBClassifier(**params)
    if name == "catboost":
        return CatBoostClassifier(**params)
    raise ValueError(f"Unknown model: {name}")

# =============================================================================
# WALK-FORWARD SPLITS (PURGED EXPANDING)
# =============================================================================

years = base_df["date"].dt.year
unique_years = sorted(years.unique())

EMBARGO_DAYS = max(config["horizons"])
splits = []

for i in range(2, len(unique_years)):
    train_years = unique_years[:i]
    test_year = unique_years[i]

    train_idx = base_df[years.isin(train_years)].index
    test_idx = base_df[years == test_year].index

    if len(train_idx) > 0:
        embargo_start = test_idx.min() - EMBARGO_DAYS
        train_idx = train_idx[train_idx < embargo_start]

    splits.append((train_idx, test_idx, test_year))

logger.info(f"Prepared {len(splits)} walk-forward splits")

# =============================================================================
# GRID SEARCH
# =============================================================================

best_params = {}

for horizon in config["horizons"]:
    logger.info(f"START Horizon={horizon}")
    best_params[horizon] = {}

    for model_name, model_cfg in config["models"].items():
        if not model_cfg["enabled"]:
            continue
        
        if model_name == "garch":
            continue

        logger.info(f"Model={model_name}")

        grid = ParameterGrid(model_cfg["param_grid"])
        model_scores = []

        for params in grid:
            fold_scores = []
            skipped_years = set()

            for train_idx, test_idx, test_year in splits:
                df_train = base_df.loc[train_idx]
                df_test  = base_df.loc[test_idx]
                
                # -------- Feature engineering --------
                X_train, X_test = engineer_features_fold(df_train, df_test)
                if X_train.empty or X_test.empty:
                    continue

                # DROP FEATURES
                DROP_COLS = [
                    "es_vol_14d", "es_vol_7d",
                    "es_ret_z_20", "es_ret", "es_range",
                    "vix_chg",
                    "dxy_ret_z",
                    "US10Y_chg_5d", "TED_SPREAD_chg_5d",
                    "pct_negative",
                    "sentiment_std", "sentiment_change", "sentiment_change_lag2",
                    "sentiment_mean_lag1", "sentiment_mean_roll_10",
                    "sentiment_std_lag1", "sentiment_zscore_10_lag2",
                ]
                X_train = X_train.drop(columns=[c for c in DROP_COLS if c in X_train.columns])
                X_test = X_test.drop(columns=[c for c in DROP_COLS if c in X_test.columns])

                # -------- Labels --------
                y = generate_stress_targets_for_fold(
                    es_df=es_df,
                    train_end=df_train["date"].max(),
                    horizon=horizon,
                )

                y_train = y.loc[X_train["date"]].dropna()
                y_test  = y.loc[X_test["date"]].dropna()

                X_tr = X_train[X_train["date"].isin(y_train.index)].drop(columns=["date"])
                X_te = X_test[X_test["date"].isin(y_test.index)].drop(columns=["date"])

                if y_train.nunique() < 2 or y_train.sum() < 5 or len(y_test) == 0:
                    if test_year not in skipped_years:
                        logger.debug(
                            f"SKIP | H{horizon} | {model_name} | "
                            f"FoldYear={test_year} | Positives={y_train.sum()}"
                        )
                        skipped_years.add(test_year)
                    continue
                
                if y_test.sum() == 0:
                    logger.debug(
                        f"SKIP-SCORE | H{horizon} | {model_name} | "
                        f"FoldYear={test_year} | No positives in y_test"
                    )
                    continue

                model_params = params.copy()

                if model_name == "xgboost" and model_params.get("scale_pos_weight") == "auto":
                    weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                    model_params["scale_pos_weight"] = min(weight, 50)

                if model_name == "catboost" and model_params.get("auto_class_weights") == "Balanced":
                    pass

                model = build_model(model_name, model_params)
                model.fit(X_tr, y_train)

                proba = model.predict_proba(X_te)[:, 1]
                score = average_precision_score(y_test, proba)
                fold_scores.append(score)

                logger.debug(
                    f"H{horizon} | {model_name} | "
                    f"FoldYear={test_year} | PR-AUC={score:.4f}"
                )

            if fold_scores:
                model_scores.append(
                    {
                        "mean_pr_auc": float(np.mean(fold_scores)),
                        "std_pr_auc": float(np.std(fold_scores)),
                        "params": model_params,
                    }
                )

        logger.info(
            f"COMPLETED | H{horizon} | {model_name} | "
            f"Evaluated {len(model_scores)} valid configs"
        )

        if not model_scores:
            logger.info(f"No valid scores for H{horizon} | {model_name}")
            continue

        best = max(model_scores, key=lambda x: x["mean_pr_auc"])
        best_params[horizon][model_name] = best

        logger.info(
            f"BEST | H{horizon} | {model_name} | "
            f"PR-AUC={best['mean_pr_auc']:.4f} ± {best['std_pr_auc']:.4f}"
        )

# =============================================================================
# SAVE BEST PARAMETERS
# =============================================================================

BEST_PARAM_PATH = Path("../artifacts/best_model_params.yaml")
BEST_PARAM_PATH.parent.mkdir(parents=True, exist_ok=True)

# Convert numpy types to Python types
def clean_for_yaml(obj):
    if isinstance(obj, dict):
        return {k: clean_for_yaml(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

best_params = clean_for_yaml(best_params)

with open(BEST_PARAM_PATH, "w") as f:
    yaml.safe_dump(best_params, f)

logger.info(f"Saved best hyperparameters -> {BEST_PARAM_PATH}")

In [ ]:
# =============================================================================
# CELL 3: TRAIN FINAL MODELS + OOF PREDICTIONS + THRESHOLD TUNING
# =============================================================================

import pickle
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
from sklearn.metrics import f1_score

MODEL_DIR = Path("../data/models")
OOF_DIR = Path("../data/oof")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLS = [
    "es_vol_14d", "es_vol_7d",     
    "es_ret_z_20", "es_ret", "es_range",
    "vix_chg",
    "dxy_ret_z",
    "US10Y_chg_5d", "TED_SPREAD_chg_5d",
    "pct_negative",
    "sentiment_std", "sentiment_change", "sentiment_change_lag2",
    "sentiment_mean_lag1", "sentiment_mean_roll_10",
    "sentiment_std_lag1", "sentiment_zscore_10_lag2",
]

# Load best params
with open("../artifacts/best_model_params.yaml", "r") as f:
    best_params = yaml.safe_load(f)

# Full training data 2019-2024
train_data = base_df[base_df["date"].dt.year <= 2024]

logger.info(f"Training final models on {len(train_data)} rows (2019-2024)")

for horizon in config["horizons"]:
    logger.info(f"Horizon={horizon}")

    for model_name in ["logistic_regression", "xgboost", "catboost"]:
        if model_name not in best_params.get(horizon, {}):
            continue

        logger.info(f"  Model={model_name}")

        # Get best params
        params = best_params[horizon][model_name]["params"]

        # Engineer features on full train set
        X_full, _ = engineer_features_fold(train_data, train_data.iloc[:0])
        X_full = X_full.drop(columns=[c for c in DROP_COLS if c in X_full.columns])

        # Generate labels
        y = generate_stress_targets_for_fold(
            es_df=es_df,
            train_end=train_data["date"].max(),
            horizon=horizon,
        )

        y_full = y.loc[X_full["date"]].dropna()
        X_tr = X_full[X_full["date"].isin(y_full.index)].drop(columns=["date"])

        # Auto scale_pos_weight for XGBoost
        if model_name == "xgboost" and params.get("scale_pos_weight") == "auto":
            weight = (y_full == 0).sum() / max((y_full == 1).sum(), 1)
            params["scale_pos_weight"] = min(weight, 50)

        # Train
        model = build_model(model_name, params)
        model.fit(X_tr, y_full)

        # Save model
        model_path = MODEL_DIR / f"{model_name}_h{horizon}.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(model, f)

        logger.info(f"    Saved -> {model_path}")

        # Generate OOF predictions
        oof_preds = []
        oof_dates = []
        oof_actuals = []

        for train_idx, test_idx, test_year in splits:
            df_train = base_df.loc[train_idx]
            df_test = base_df.loc[test_idx]

            X_train, X_test = engineer_features_fold(df_train, df_test)
            if X_train.empty or X_test.empty:
                continue

            X_train = X_train.drop(columns=[c for c in DROP_COLS if c in X_train.columns])
            X_test = X_test.drop(columns=[c for c in DROP_COLS if c in X_test.columns])

            y = generate_stress_targets_for_fold(
                es_df=es_df,
                train_end=df_train["date"].max(),
                horizon=horizon,
            )

            y_train = y.loc[X_train["date"]].dropna()
            y_test = y.loc[X_test["date"]].dropna()

            X_tr = X_train[X_train["date"].isin(y_train.index)].drop(columns=["date"])
            X_te = X_test[X_test["date"].isin(y_test.index)].drop(columns=["date"])

            if y_train.nunique() < 2 or len(y_test) == 0:
                continue

            # Train fold model
            fold_params = params.copy()
            if model_name == "xgboost" and fold_params.get("scale_pos_weight") == "auto":
                weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                fold_params["scale_pos_weight"] = min(weight, 50)

            fold_model = build_model(model_name, fold_params)
            fold_model.fit(X_tr, y_train)

            proba = fold_model.predict_proba(X_te)[:, 1]

            oof_preds.extend(proba)
            oof_dates.extend(X_test[X_test["date"].isin(y_test.index)]["date"])
            oof_actuals.extend(y_test)

        # Save OOF
        oof_df = pd.DataFrame({
            "date": oof_dates,
            "actual": oof_actuals,
            "predicted": oof_preds,
        })

        oof_path = OOF_DIR / f"{model_name}_h{horizon}_oof.csv"
        oof_df.to_csv(oof_path, index=False)
        logger.info(f"    OOF saved -> {oof_path} ({len(oof_df)} predictions)")

logger.info("Final training complete")

# =============================================================================
# THRESHOLD TUNING FROM OOF
# =============================================================================

optimal_thresholds = {}

for horizon in config["horizons"]:
    optimal_thresholds[horizon] = {}

    for model_name in ["logistic_regression", "xgboost", "lightgbm", "catboost"]:
        oof_path = OOF_DIR / f"{model_name}_h{horizon}_oof.csv"
        if not oof_path.exists():
            continue

        oof = pd.read_csv(oof_path)
        y_true = oof["actual"].values
        y_prob = oof["predicted"].values

        best_t, best_f1 = 0.5, -1.0
        for t in np.linspace(0.01, 0.99, 99):
            score = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t = float(np.round(t, 2))

        optimal_thresholds[horizon][model_name] = best_t
        logger.info(
            f"H{horizon} | {model_name} | Best threshold={best_t:.2f} | OOF F1={best_f1:.3f}"
        )

print(optimal_thresholds)

In [ ]:
# =============================================================================
# CELL 4: TEST ON 2025 (OUT-OF-SAMPLE)
# =============================================================================

import pickle
import os
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, recall_score, f1_score

DROP_COLS = [
    "es_vol_14d", "es_vol_7d",      
    "es_ret_z_20", "es_ret", "es_range",
    "vix_chg",
    "dxy_ret_z",
    "US10Y_chg_5d", "TED_SPREAD_chg_5d",
    "pct_negative",
    "sentiment_std", "sentiment_change", "sentiment_change_lag2",
    "sentiment_mean_lag1", "sentiment_mean_roll_10",
    "sentiment_std_lag1", "sentiment_zscore_10_lag2",
]

OOS_DIR = "../artifacts/ml_models_oos"
os.makedirs(OOS_DIR, exist_ok=True)

test_data = base_df[
    (base_df["date"] >= "2025-01-01") &
    (base_df["date"] <= "2026-01-01")
]
logger.info(f"Testing on {len(test_data)} rows (2025)")

results = []

for horizon in config["horizons"]:
    logger.info(f"Horizon={horizon}")

    for model_name in ["logistic_regression", "xgboost", "catboost"]:

        model_path = MODEL_DIR / f"{model_name}_h{horizon}.pkl"
        if not model_path.exists():
            continue

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        # Recreate test features exactly as training pipeline
        _, X_test = engineer_features_fold(
            base_df[base_df["date"].dt.year <= 2024],
            test_data
        )

        X_test = X_test.drop(
            columns=[c for c in DROP_COLS if c in X_test.columns]
        )

        if X_test.empty:
            continue

        y = generate_stress_targets_for_fold(
            es_df=es_df,
            train_end=base_df[base_df["date"].dt.year <= 2024]["date"].max(),
            horizon=horizon,
        )

        y_test = y.loc[X_test["date"]].dropna()
        X_te = X_test[X_test["date"].isin(y_test.index)].copy()

        if len(y_test) == 0 or y_test.sum() == 0:
            continue

        X_te_model = X_te.drop(columns=["date"])

        proba = model.predict_proba(X_te_model)[:, 1]

        threshold = optimal_thresholds[horizon][model_name]
        y_pred = (proba >= threshold).astype(int)

        # =========================
        # SAVE DAILY PREDICTIONS
        # =========================
        pred_df = pd.DataFrame({
            "date": X_te["date"].values,
            "y_true": y_test.values,
            "proba": proba,
            "pred": y_pred
        })

        pred_df.to_csv(
            f"{OOS_DIR}/{model_name}_oos_predictions_h{horizon}.csv",
            index=False
        )

        # =========================
        # METRICS
        # =========================
        pr_auc = average_precision_score(y_test, proba)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)

        top_5_idx = np.argsort(proba)[-int(0.05 * len(proba)):]
        top_10_idx = np.argsort(proba)[-int(0.10 * len(proba)):]

        prec_5 = y_test.iloc[top_5_idx].mean()
        prec_10 = y_test.iloc[top_10_idx].mean()

        results.append({
            "horizon": horizon,
            "model": model_name,
            "threshold": threshold,
            "pr_auc": pr_auc,
            "f1": f1,
            "recall": rec,
            "prec@5%": prec_5,
            "prec@10%": prec_10,
        })

        logger.info(
            f"  {model_name}: t={threshold} | PR-AUC={pr_auc:.3f} | "
            f"F1={f1:.3f} | Recall={rec:.3f} | "
            f"Prec@5%={prec_5:.3f} | Prec@10%={prec_10:.3f}"
        )

results_df = pd.DataFrame(results)
results_df.to_csv("../artifacts/ml_models_oos/oos_results_2025.csv", index=False)

logger.info(f"Saved -> ../artifacts//ml_models_oosoos_results_2025.csv")
print("\n" + "=" * 60)
print(results_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 5: FEATURE IMPORTANCE & EXPLAINABILITY
# =============================================================================

import pickle
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# =============================================================================
#  GLOBAL STYLE  —  light academic palette
# =============================================================================
BG          = "#FFFFFF"
PANEL_BG    = "#F7F8FA"
BORDER      = "#D0D5DD"
TEXT_PRI    = "#1A1D23"
TEXT_MUT    = "#6B7280"
ACCENT_BLUE = "#2563EB"
ACCENT_RED  = "#DC2626"
ACCENT_AMB  = "#D97706"
ACCENT_GRN  = "#16A34A"
ACCENT_PUR  = "#7C3AED"

MODEL_COLORS = {
    "logistic_regression": ACCENT_RED,
    "xgboost":             ACCENT_GRN,
    "catboost":            ACCENT_BLUE,
}
MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "xgboost":             "XGBoost",
    "catboost":            "CatBoost",
}

plt.rcParams.update({
    "figure.facecolor":   BG,
    "axes.facecolor":     PANEL_BG,
    "axes.edgecolor":     BORDER,
    "axes.labelcolor":    TEXT_PRI,
    "axes.titlecolor":    TEXT_PRI,
    "axes.grid":          True,
    "grid.color":         BORDER,
    "grid.linewidth":     0.55,
    "grid.alpha":         0.7,
    "grid.linestyle":     "--",
    "xtick.color":        TEXT_MUT,
    "ytick.color":        TEXT_MUT,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.titlepad":      10,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":        "DejaVu Sans",
    "text.color":         TEXT_PRI,
    "legend.facecolor":   BG,
    "legend.edgecolor":   BORDER,
    "legend.fontsize":    9,
    "savefig.facecolor":  BG,
    "savefig.dpi":        180,
    "figure.dpi":         110,
})


def _watermark(fig):
    fig.text(0.99, 0.005, "Confidential Research", fontsize=7,
             color=TEXT_MUT, ha="right", va="bottom", alpha=0.5, style="italic")


def _spine(ax):
    ax.spines["bottom"].set_color(BORDER)
    ax.spines["left"].set_color(BORDER)


# =============================================================================
#  UNCHANGED SETUP LOGIC
# =============================================================================
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
DROP_COLS = [
    "es_vol_14d", "es_vol_7d",
    "es_ret_z_20", "es_ret", "es_range",
    "vix_chg",
    "dxy_ret_z",
    "US10Y_chg_5d", "TED_SPREAD_chg_5d",
    "pct_negative",
    "sentiment_std", "sentiment_change", "sentiment_change_lag2",
    "sentiment_mean_lag1", "sentiment_mean_roll_10",
    "sentiment_std_lag1", "sentiment_zscore_10_lag2",
]

logger.info("Starting feature importance analysis")

train_data = base_df[base_df["date"].dt.year <= 2024]
X_full, _ = engineer_features_fold(train_data, train_data.iloc[:0])
X_full = X_full.drop(columns=[c for c in DROP_COLS if c in X_full.columns])
feature_names = [col for col in X_full.columns if col != "date"]
logger.info(f"Total features: {len(feature_names)}")

test_data = base_df[base_df["date"].dt.year == 2025]
_, X_test = engineer_features_fold(train_data, test_data)
X_test = X_test.drop(columns=[c for c in DROP_COLS if c in X_test.columns])

for horizon in config["horizons"]:
    logger.info(f"Horizon={horizon}")

    y = generate_stress_targets_for_fold(
        es_df=es_df,
        train_end=train_data["date"].max(),
        horizon=horizon,
    )
    y_test = y.loc[X_test["date"]].dropna()
    X_te = (
        X_test[X_test["date"].isin(y_test.index)]
        .drop(columns=["date"])
        .reset_index(drop=True)
    )

    for model_name in ["logistic_regression", "xgboost", "catboost"]:
        logger.info(f"  Model={model_name}")

        model_path = MODEL_DIR / f"{model_name}_h{horizon}.pkl"
        if not model_path.exists():
            logger.warning(f"    Model not found: {model_path}")
            continue

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        # ------------------------------------------------------------------
        # Feature importance  (unchanged logic)
        # ------------------------------------------------------------------
        if model_name == "logistic_regression":
            importances = np.abs(model.coef_[0])
        else:
            importances = model.feature_importances_

        importance_df = (
            pd.DataFrame({"feature": feature_names, "importance": importances})
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )

        imp_path = PLOT_DIR / f"{model_name}_h{horizon}_importance.csv"
        importance_df.to_csv(imp_path, index=False)
        logger.info(f"    Saved importance -> {imp_path}")

        # ── Top-10 bar chart ──────────────────────────────────────────────
        top10 = importance_df.head(10)
        col   = MODEL_COLORS.get(model_name, ACCENT_BLUE)
        lbl   = MODEL_LABELS.get(model_name, model_name)

        norm_imp = top10["importance"].values / top10["importance"].max()
        bar_rgba = [plt.matplotlib.colors.to_rgba(col, alpha=0.30 + 0.60 * n)
                    for n in norm_imp]

        fig, ax = plt.subplots(figsize=(10, 6))
        fig.subplots_adjust(top=0.84, bottom=0.10, left=0.28,
                            right=0.97)

        bars = ax.barh(range(10), top10["importance"].values,
                       color=bar_rgba, edgecolor="white",
                       linewidth=0.6, zorder=3, height=0.62)

        # Inline value labels
        x_max = top10["importance"].max()
        for bar, val in zip(bars, top10["importance"].values):
            ax.text(val + x_max * 0.012, bar.get_y() + bar.get_height() / 2,
                    f"{val:.4f}", va="center", ha="left",
                    fontsize=8, color=TEXT_MUT)

        ax.set_yticks(range(10))
        ax.set_yticklabels(top10["feature"].values, fontsize=9.5)
        ax.invert_yaxis()
        ax.set_xlabel("Importance Score", labelpad=6)
        ax.set_xlim(0, x_max * 1.20)
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda v, _: f"{v:.3f}"))
        ax.set_title(f"Top 10 Features - {lbl}  |  {horizon}-Day Horizon")
        ax.grid(axis="x", color=BORDER, linewidth=0.55,
                linestyle="--", alpha=0.7)
        ax.grid(axis="y", visible=False)
        _spine(ax)

        plot_path = PLOT_DIR / f"{model_name}_h{horizon}_top10.png"
        fig.savefig(plot_path, bbox_inches="tight")
        plt.close(fig)
        logger.info(f"    Saved plot -> {plot_path}")

        # ------------------------------------------------------------------
        # SHAP  (unchanged logic)
        # ------------------------------------------------------------------
        X_test_aligned = (
            X_test[X_test["date"].isin(y_test.index)]
            .copy()
            .reset_index(drop=True)
        )

        X_all_with_dates = X_test_aligned.copy()
        X_all = X_all_with_dates.drop(columns=["date"])

        if model_name == "logistic_regression":
            explainer = shap.LinearExplainer(
                model, X_te, feature_perturbation="interventional"
            )
        else:
            explainer = shap.TreeExplainer(model)

        shap_values = explainer.shap_values(X_all)

        # Save full OOS SHAP values for dashboard
        shap_df = pd.DataFrame(shap_values, columns=feature_names)
        shap_df.insert(0, "date", X_all_with_dates["date"].values)
        shap_df.to_csv(PLOT_DIR / f"{model_name}_h{horizon}_shap_values.csv", index=False)

        # SHAP summary plot
        fig, ax = plt.subplots(figsize=(10, 8))
        fig.subplots_adjust(top=0.84, bottom=0.10, left=0.28, right=0.90)

        shap.summary_plot(
            shap_values,
            X_all,
            feature_names=feature_names,
            show=False,
            plot_size=None,
            color_bar=True,
            alpha=0.75,
        )

        ax = plt.gca()
        ax.set_facecolor(PANEL_BG)
        fig.patch.set_facecolor(BG)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_color(BORDER)
        ax.spines["left"].set_color(BORDER)
        ax.tick_params(colors=TEXT_MUT, labelsize=9)
        ax.set_xlabel("SHAP Value  (impact on model output)", color=TEXT_PRI, fontsize=10, labelpad=6)
        ax.set_title(f"SHAP Summary - {lbl}  |  {horizon}-Day Horizon",
                    fontsize=14, fontweight="bold", color=TEXT_PRI, pad=10)

        shap_path = PLOT_DIR / f"{model_name}_h{horizon}_shap.png"
        fig.savefig(shap_path, bbox_inches="tight")
        plt.close(fig)
        logger.info(f"    Saved SHAP -> {shap_path}")

logger.info("Feature importance analysis complete")

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os
from itertools import product
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from utils.labels import generate_stress_targets_for_fold
from utils.garch_model import run_garch_fold
import warnings

warnings.filterwarnings("ignore")

# =========================
# LOAD DATA
# =========================
es_df = pd.read_csv("../data/processed/ES_D.csv", parse_dates=["date"])
es_df = es_df.sort_values("date").reset_index(drop=True)

# =========================
# LOAD YAML CONFIG
# =========================
with open("../config/model_config.yaml", "r") as f:
    config = yaml.safe_load(f)

garch_grid = config["models"]["garch"]["param_grid"]
p_list    = garch_grid["p"]
q_list    = garch_grid["q"]
dist_list = garch_grid["distribution"]

# =========================
# CREATE EXPANDING SPLITS
# =========================
def create_expanding_splits(df):
    splits = []
    start = df["date"].min()
    for i in range(4):
        train_end  = start + pd.DateOffset(years=2 + i)
        test_start = train_end + pd.DateOffset(days=1)
        test_end   = test_start + pd.DateOffset(years=1)
        splits.append((start, train_end, test_start, test_end))
    return splits

splits = create_expanding_splits(es_df)

# =========================
# GRID SEARCH
# =========================
horizons    = [1, 3]
all_results = []
best_configs = {}

for horizon in horizons:

    best_score  = -np.inf
    best_params = None

    for p, q, dist in product(p_list, q_list, dist_list):

        fold_scores = []

        for train_start, train_end, test_start, test_end in splits:

            stress_series = generate_stress_targets_for_fold(
                es_df=es_df, train_end=train_end, horizon=horizon
            )
            y_test = stress_series.loc[
                (stress_series.index >= test_start) &
                (stress_series.index <= test_end)
            ]

            if len(y_test) == 0 or y_test.nunique() < 2:
                continue

            # OOF calibration: last year of training window
            oof_train_end  = train_end - pd.DateOffset(years=2)
            oof_test_start = oof_train_end + pd.DateOffset(days=1)

            try:
                oof_out = run_garch_fold(
                    es_df=es_df,
                    train_end=oof_train_end,
                    test_start=oof_test_start,
                    test_end=train_end,
                    horizon=horizon,
                    p=p, q=q, distribution=dist
                )
            except Exception:
                continue

            y_oof = stress_series.loc[
                (stress_series.index >= oof_out["date"].min()) &
                (stress_series.index <= oof_out["date"].max())
            ]
            oof_merged = pd.merge(
                oof_out, y_oof.rename("y_true"),
                left_on="date", right_index=True, how="inner"
            ).dropna()

            if oof_merged["y_true"].nunique() < 2:
                continue

            # fit Platt scaler on OOF
            calibrator = Pipeline([
                ("scaler", StandardScaler()),
                ("lr", LogisticRegression(class_weight="balanced"))
            ])
            calibrator.fit(
                oof_merged[["garch_vol_forecast"]].values,
                oof_merged["y_true"].values
            )

            # test fold forecasts
            try:
                garch_out = run_garch_fold(
                    es_df=es_df,
                    train_end=train_end,
                    test_start=test_start,
                    test_end=test_end,
                    horizon=horizon,
                    p=p, q=q, distribution=dist
                )
            except Exception:
                continue

            merged = pd.merge(
                garch_out, y_test.rename("y_true"),
                left_on="date", right_index=True, how="inner"
            ).dropna()

            if len(merged) == 0 or merged["y_true"].nunique() < 2:
                continue

            proba = calibrator.predict_proba(
                merged[["garch_vol_forecast"]].values
            )[:, 1]

            fold_scores.append(average_precision_score(merged["y_true"], proba))

        if not fold_scores:
            print(f"No valid folds for H{horizon} p={p} q={q} dist={dist}")
            continue

        mean_score = float(np.mean(fold_scores))
        all_results.append({
            "horizon": horizon, "p": p, "q": q,
            "distribution": dist, "mean_pr_auc": mean_score
        })

        if mean_score > best_score:
            best_score  = mean_score
            best_params = {"p": p, "q": q, "distribution": dist, "mean_pr_auc": mean_score}

    best_configs[f"horizon_{horizon}"] = best_params

# =========================
# SAVE BEST CONFIG
# =========================
os.makedirs("../artifacts", exist_ok=True)
with open("../artifacts/best_garch_param.yaml", "w") as f:
    yaml.dump(best_configs, f)

# =========================
# COLLECT OOF FORECASTS
# =========================
os.makedirs("../artifacts/garch_oof", exist_ok=True)

for horizon in horizons:

    params = best_configs[f"horizon_{horizon}"]
    if params is None:
        continue

    oof_rows = []

    for train_start, train_end, test_start, test_end in splits:

        stress_series = generate_stress_targets_for_fold(
            es_df=es_df, train_end=train_end, horizon=horizon
        )
        y_test = stress_series.loc[
            (stress_series.index >= test_start) &
            (stress_series.index <= test_end)
        ]

        if len(y_test) == 0:
            continue

        try:
            garch_out = run_garch_fold(
                es_df=es_df,
                train_end=train_end,
                test_start=test_start,
                test_end=test_end,
                horizon=horizon,
                p=params["p"], q=params["q"],
                distribution=params["distribution"]
            )
        except Exception:
            continue

        merged = pd.merge(
            garch_out[["date", "garch_vol_forecast"]],
            y_test.rename("y_true"),
            left_on="date", right_index=True, how="inner"
        )
        oof_rows.append(merged)

    if oof_rows:
        oof_df = pd.concat(oof_rows).sort_values("date").reset_index(drop=True)
        oof_df.to_csv(f"../artifacts/garch_oof/garch_oof_h{horizon}.csv", index=False)
        print(f"H{horizon}: saved {len(oof_df)} OOF rows")

print(pd.DataFrame(all_results).sort_values("mean_pr_auc", ascending=False))

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from arch import arch_model
from utils.labels import generate_stress_targets_for_fold
import warnings

warnings.filterwarnings("ignore")

# =========================
# LOAD DATA
# =========================
es_df = pd.read_csv("../data/processed/ES_D.csv", parse_dates=["date"])
es_df = es_df.sort_values("date").reset_index(drop=True)
es_df["ret"] = np.log(es_df["close"]).diff()
es_df = es_df.dropna().reset_index(drop=True)

# =========================
# LOAD BEST PARAMETERS
# =========================
with open("../artifacts/best_garch_param.yaml", "r") as f:
    best_cfg = yaml.safe_load(f)

# =========================
# TRAIN / OOS SPLIT
# =========================
train_end_date = pd.Timestamp("2024-12-31")
train_df = es_df[es_df["date"] <= train_end_date].copy()
oos_df   = es_df[es_df["date"] > train_end_date].copy()

os.makedirs("../artifacts/garch_oos", exist_ok=True)

horizons = [1, 3]
oos_results = []

for horizon in horizons:

    params = best_cfg[f"horizon_{horizon}"]
    p    = params["p"]
    q    = params["q"]
    dist = params["distribution"]

    # -------------------------------------------------------
    # FIT CALIBRATOR on OOF vol forecasts
    # -------------------------------------------------------
    oof_path = f"../artifacts/garch_oof/garch_oof_h{horizon}.csv"
    if not os.path.exists(oof_path):
        raise FileNotFoundError(
            f"OOF file not found: {oof_path}. Run garch_grid_search.py first."
        )

    oof_df = pd.read_csv(oof_path, parse_dates=["date"]).dropna()

    calibrator = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(class_weight="balanced"))
    ])
    calibrator.fit(
        oof_df[["garch_vol_forecast"]].values,
        oof_df["y_true"].values
    )

    # -------------------------------------------------------
    # GENERATE OOS VOL FORECASTS (expanding)
    # -------------------------------------------------------
    stress_series = generate_stress_targets_for_fold(
        es_df=es_df,
        train_end=train_end_date,
        horizon=horizon
    )
    y_oos = stress_series.loc[oos_df["date"]]

    expanding_returns = train_df["ret"].values
    forecasts = []

    for i in range(len(oos_df)):
        try:
            model = arch_model(
                expanding_returns,
                mean="Zero", vol="GARCH",
                p=p, q=q, dist=dist, rescale=False
            )
            fitted = model.fit(disp="off")
            var = fitted.forecast(horizon=horizon, reindex=False).variance.values[-1, horizon - 1]
            vol = np.sqrt(var)
        except Exception:
            vol = np.nan

        forecasts.append(vol)
        expanding_returns = np.append(expanding_returns, oos_df["ret"].iloc[i])

    oos_pred = pd.DataFrame({
        "date": oos_df["date"].values,
        "garch_vol_forecast": forecasts
    }).dropna()

    vol_threshold = train_df["ret"].rolling(14).std().quantile(0.90)
    oos_pred["proba"] = calibrator.predict_proba(
        oos_pred[["garch_vol_forecast"]].values
    )[:, 1]
    oos_pred["pred"] = (oos_pred["garch_vol_forecast"] >= vol_threshold).astype(int)

    merged = pd.merge(
        oos_pred, y_oos.rename("y_true"),
        left_on="date", right_index=True, how="inner"
    ).dropna(subset=["y_true", "proba"])

    merged.to_csv(
        f"../artifacts/garch_oos/garch_oos_predictions_h{horizon}.csv",
        index=False
    )

    if len(merged) == 0 or merged["y_true"].nunique() < 2:
        continue

    oos_results.append({
        "horizon":     horizon,
        "p": p, "q": q, "distribution": dist,
        "oos_pr_auc":  average_precision_score(merged["y_true"], merged["proba"]),
        "oos_roc_auc": roc_auc_score(merged["y_true"], merged["proba"]),
        "oos_f1":      f1_score(merged["y_true"], merged["pred"])
    })

    print(f"H{horizon} proba stats:")
    print(oos_pred["proba"].describe())

metrics_df = pd.DataFrame(oos_results)
metrics_df.to_csv("../artifacts/garch_oos/garch_oos_metrics.csv", index=False)
print(metrics_df)

In [ ]:
import pandas as pd

# =========================
# LOAD OOS METRICS
# =========================

ml_metrics = pd.read_csv("../artifacts/ml_models_oos/oos_results_2025.csv")
garch_metrics = pd.read_csv("../artifacts/garch_oos/garch_oos_metrics.csv")
# naive_metrics = pd.read_csv("../artifacts/naive_baselines_oos/baseline_oos_metrics.csv")

# =========================
# STANDARDIZE COLUMN NAMES
# =========================

# ML
ml_metrics = ml_metrics.rename(columns={
    "pr_auc": "oos_pr_auc",
    "f1": "oos_f1",
    "recall": "oos_recall"
})
ml_metrics["model"] = ml_metrics["model"]

# GARCH already correct structure
garch_metrics["model"] = "garch"

# Naive already has model column

# =========================
# ALIGN COMMON COLUMNS
# =========================

common_cols = [
    "model", "horizon",
    "oos_pr_auc",
    "oos_f1", "oos_recall",
    "prec@5%", "prec@10%"
]

# ML is missing oos_roc_auc — add it as NaN
ml_metrics["oos_roc_auc"] = None

ml_final = ml_metrics[common_cols]
garch_final = garch_metrics.reindex(columns=common_cols)  # fills missing with NaN

final_table = pd.concat([ml_final, garch_final], ignore_index=True)

final_table = final_table.sort_values(
    ["horizon", "oos_pr_auc"],
    ascending=[True, False]
).reset_index(drop=True)

final_table

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.metrics import brier_score_loss

# =========================
# COLLECT ALL PREDICTION FILES
# =========================

paths = []
paths += glob.glob("../artifacts/ml_models_oos/*.csv")
paths += glob.glob("../artifacts/garch_oos/*.csv")
#paths += glob.glob("../artifacts/naive_baselines_oos/*.csv")

calibration_rows = []

for path in paths:

    df = pd.read_csv(path)

    # Must contain probability
    if "proba" not in df.columns:
        continue

    # Detect target column
    if "y_true" in df.columns:
        y_col = "y_true"
    elif "stress" in df.columns:
        y_col = "stress"
    else:
        continue

    df = df.dropna(subset=[y_col, "proba"])

    if df[y_col].nunique() < 2:
        continue

    # Extract filename cleanly
    filename = os.path.basename(path).replace(".csv", "")

    # Extract horizon
    if "h1" in filename:
        horizon = 1
    elif "h3" in filename:
        horizon = 3
    else:
        continue

    # Model name = everything before "_oos_predictions"
    if "_oos_predictions" in filename:
        model = filename.split("_oos_predictions")[0]
    else:
        model = filename

    # Clip probabilities into [0,1]
    proba = np.clip(df["proba"].values, 0, 1)
    y_true = df[y_col].values

    brier = brier_score_loss(y_true, proba)

    calibration_rows.append({
        "model": model,
        "horizon": horizon,
        "brier_score": brier
    })

calibration_df = pd.DataFrame(calibration_rows)

calibration_df.sort_values(["horizon", "brier_score"]).reset_index(drop=True)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from sklearn.calibration import calibration_curve

# =============================================================================
#  GLOBAL STYLE  —  light academic palette
# =============================================================================
BG          = "#FFFFFF"
PANEL_BG    = "#F7F8FA"
BORDER      = "#D0D5DD"
TEXT_PRI    = "#1A1D23"
TEXT_MUT    = "#6B7280"
ACCENT_BLUE = "#2563EB"
ACCENT_RED  = "#DC2626"
ACCENT_AMB  = "#D97706"
ACCENT_GRN  = "#16A34A"
ACCENT_PUR  = "#7C3AED"

MODEL_COLORS = {
    "garch":               ACCENT_AMB,
    "logistic_regression": ACCENT_BLUE,
    "xgboost":             ACCENT_GRN,
    "catboost":            ACCENT_RED,
}
MODEL_MARKERS = {
    "garch":               "D",
    "logistic_regression": "o",
    "xgboost":             "s",
    "catboost":            "^",
}
MODEL_LABELS = {
    "garch":               "GARCH",
    "logistic_regression": "Logistic Regression",
    "xgboost":             "XGBoost",
    "catboost":            "CatBoost",
}

plt.rcParams.update({
    "figure.facecolor":   BG,
    "axes.facecolor":     PANEL_BG,
    "axes.edgecolor":     BORDER,
    "axes.labelcolor":    TEXT_PRI,
    "axes.titlecolor":    TEXT_PRI,
    "axes.grid":          True,
    "grid.color":         BORDER,
    "grid.linewidth":     0.55,
    "grid.alpha":         0.7,
    "grid.linestyle":     "--",
    "xtick.color":        TEXT_MUT,
    "ytick.color":        TEXT_MUT,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.titlepad":      10,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":        "DejaVu Sans",
    "text.color":         TEXT_PRI,
    "legend.facecolor":   BG,
    "legend.edgecolor":   BORDER,
    "legend.fontsize":    9,
    "savefig.facecolor":  BG,
    "savefig.dpi":        180,
    "figure.dpi":         110,
})


def _watermark(fig):
    fig.text(0.99, 0.005, "Confidential Research", fontsize=7,
             color=TEXT_MUT, ha="right", va="bottom", alpha=0.5, style="italic")


def _spine(ax):
    ax.spines["bottom"].set_color(BORDER)
    ax.spines["left"].set_color(BORDER)


# =============================================================================
#  UNCHANGED LOGIC
# =============================================================================
ML_DIR    = "../artifacts/ml_models_oos"
GARCH_DIR = "../artifacts/garch_oos"
ML_MODELS = ["logistic_regression", "xgboost", "catboost"]


def load_pred(path):
    df = pd.read_csv(path, parse_dates=["date"])
    if "stress" in df.columns and "y_true" not in df.columns:
        df = df.rename(columns={"stress": "y_true"})
    return df


def merge_models(h):
    dfs = []
    for name, folder in [("garch", GARCH_DIR)]:
        path = os.path.join(folder, f"{name}_oos_predictions_h{h}.csv")
        if os.path.exists(path):
            d = load_pred(path).rename(columns={"proba": f"proba_{name}",
                                                 "pred":  f"pred_{name}"})
            dfs.append(d)
    for m in ML_MODELS:
        path = os.path.join(ML_DIR, f"{m}_oos_predictions_h{h}.csv")
        if os.path.exists(path):
            d = load_pred(path).rename(columns={"proba": f"proba_{m}",
                                                 "pred":  f"pred_{m}"})
            dfs.append(d)
    if not dfs:
        return None
    out = dfs[0]
    for d in dfs[1:]:
        out = out.merge(d, on="date", how="outer", suffixes=("", "_dup"))
        if "y_true_dup" in out.columns:
            out["y_true"] = out["y_true"].combine_first(out["y_true_dup"])
            out = out.drop(columns=["y_true_dup"])
    return out.sort_values("date").reset_index(drop=True)


data = []
for h in [1, 3]:
    merged = merge_models(h)
    if merged is None:
        continue
    for c in [col for col in merged.columns if col.startswith("proba_")]:
        subset = merged[["y_true", c]].dropna()
        data.append((
            c.replace("proba_", ""),
            h,
            subset["y_true"].values,
            np.clip(subset[c].values, 0, 1),
        ))

# =============================================================================
#  PLOTS
# =============================================================================
for horizon in [1, 3]:
    horizon_data = [(m, h, yt, p) for m, h, yt, p in data if h == horizon]
    if not horizon_data:
        continue

    fig, ax = plt.subplots(figsize=(8, 7))
    fig.subplots_adjust(top=0.84, bottom=0.11, left=0.10, right=0.97)

    # Perfect calibration reference
    ax.plot([0, 1], [0, 1],
            linestyle="--", linewidth=1.4,
            color=BORDER, zorder=2, label="Perfect calibration")

    # Shaded "over / under confidence" regions
    ax.fill_between([0, 1], [0, 1], [1, 1],
                alpha=0.03, color="#6B7280", zorder=1)
    ax.fill_between([0, 1], [0, 0], [0, 1],
                    alpha=0.03, color="#6B7280", zorder=1)
    ax.text(0.72, 0.30, "Over-confident", fontsize=7.5,
        color=ACCENT_RED, alpha=0.55, rotation=38, ha="center")
    ax.text(0.28, 0.72, "Under-confident", fontsize=7.5,
            color=ACCENT_GRN, alpha=0.55, rotation=38, ha="center")

    for model, h, y_true, proba in horizon_data:
        frac_pos, mean_pred = calibration_curve(
            y_true, proba, n_bins=5, strategy="quantile")

        col = MODEL_COLORS.get(model, TEXT_MUT)
        mkr = MODEL_MARKERS.get(model, "o")
        lbl = MODEL_LABELS.get(model, model)

        # Shaded area between curve and diagonal
        ax.fill_between(mean_pred, frac_pos, mean_pred,
                        alpha=0.08, color=col, zorder=3)

        ax.plot(mean_pred, frac_pos,
                marker=mkr, linewidth=2.0, markersize=7,
                color=col, zorder=5, label=lbl,
                markeredgecolor=BG, markeredgewidth=0.8)

        # Annotate each calibration bin
        for xp, yp in zip(mean_pred, frac_pos):
            ax.annotate(f"{yp:.2f}",
                        xy=(xp, yp),
                        xytext=(0, 7), textcoords="offset points",
                        ha="center", fontsize=7, color=col, alpha=0.85)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
    ax.set_xlabel("Mean Predicted Probability", labelpad=7)
    ax.set_ylabel("Observed Frequency",         labelpad=7)
    ax.set_title(f"Reliability Diagram - {horizon}-Day Horizon")
    ax.legend(loc="upper left", framealpha=0.95,
              edgecolor=BORDER, fontsize=9)
    _spine(ax)

    plt.savefig(f"../data/visuals/reliability_curve_h{horizon}.png", bbox_inches="tight")
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import glob
from sklearn.metrics import average_precision_score

np.random.seed(42)

N_BOOT = 1000

paths = []
paths += glob.glob("../artifacts/ml_models_oos/*.csv")
paths += glob.glob("../artifacts/garch_oos/*.csv")
#paths += glob.glob("../artifacts/naive_baselines_oos/*.csv")

rows = []

for path in paths:

    df = pd.read_csv(path)

    if "proba" not in df.columns:
        continue

    if "y_true" in df.columns:
        y_col = "y_true"
    elif "stress" in df.columns:
        y_col = "stress"
    else:
        continue

    df = df.dropna(subset=[y_col, "proba"])

    if df[y_col].nunique() < 2:
        continue

    filename = path.split("/")[-1].replace(".csv", "")

    if "h1" in filename:
        horizon = 1
    elif "h3" in filename:
        horizon = 3
    else:
        continue

    if "_oos_predictions" in filename:
        model = filename.split("_oos_predictions")[0]
    else:
        model = filename

    y = df[y_col].values
    p = np.clip(df["proba"].values, 0, 1)

    boot_scores = []

    n = len(y)

    for _ in range(N_BOOT):
        idx = np.random.choice(n, n, replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        score = average_precision_score(y[idx], p[idx])
        boot_scores.append(score)

    if len(boot_scores) == 0:
        continue

    mean_score = np.mean(boot_scores)
    lower = np.percentile(boot_scores, 2.5)
    upper = np.percentile(boot_scores, 97.5)

    rows.append({
        "model": model,
        "horizon": horizon,
        "pr_auc_mean": mean_score,
        "ci_lower": lower,
        "ci_upper": upper
    })

ci_df = pd.DataFrame(rows).sort_values(
    ["horizon", "pr_auc_mean"],
    ascending=[True, False]
).reset_index(drop=True)

ci_df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from pathlib import Path

# =============================================================================
#  GLOBAL STYLE
# =============================================================================
BG          = "#FFFFFF"
PANEL_BG    = "#F7F8FA"
BORDER      = "#D0D5DD"
TEXT_PRI    = "#1A1D23"
TEXT_MUT    = "#6B7280"
ACCENT_BLUE = "#2563EB"
ACCENT_RED  = "#DC2626"
ACCENT_AMB  = "#D97706"
ACCENT_GRN  = "#16A34A"

plt.rcParams.update({
    "figure.facecolor":   BG,
    "axes.facecolor":     PANEL_BG,
    "axes.edgecolor":     BORDER,
    "axes.labelcolor":    TEXT_PRI,
    "axes.titlecolor":    TEXT_PRI,
    "axes.grid":          True,
    "grid.color":         BORDER,
    "grid.linewidth":     0.55,
    "grid.alpha":         0.7,
    "grid.linestyle":     "--",
    "xtick.color":        TEXT_MUT,
    "ytick.color":        TEXT_MUT,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.titlepad":      10,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":        "DejaVu Sans",
    "text.color":         TEXT_PRI,
    "legend.facecolor":   BG,
    "legend.edgecolor":   BORDER,
    "legend.fontsize":    9,
    "savefig.facecolor":  BG,
    "savefig.dpi":        180,
    "figure.dpi":         110,
})

MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "xgboost":             "XGBoost",
    "catboost":            "CatBoost",
}

def _spine(ax):
    ax.spines["bottom"].set_color(BORDER)
    ax.spines["left"].set_color(BORDER)

def _date_axis(ax, interval=6):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# =============================================================================
#  PATHS
# =============================================================================
ML_OOS_DIR    = Path("../artifacts/ml_models_oos")
GARCH_OOS_DIR = Path("../artifacts/garch_oos")
PLOT_DIR      = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
#  LOAD RESULTS & THRESHOLDS
# =============================================================================
oos_results = pd.read_csv(ML_OOS_DIR / "oos_results_2025.csv")
thresholds  = oos_results.set_index(["model", "horizon"])["threshold"].to_dict()

# =============================================================================
#  MANUALLY SET MODEL PER HORIZON
# =============================================================================
best_per_horizon = {
    1: "catboost",
    3: "catboost"
}

# =============================================================================
#  PLOT
# =============================================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)
fig.subplots_adjust(top=0.88, bottom=0.10, left=0.07,
                    right=0.97, hspace=0.52)

for ax, horizon in zip(axes, [1, 3]):

    best_model = best_per_horizon[horizon]
    best_label = MODEL_LABELS.get(best_model, best_model)
    thresh     = thresholds[(best_model, horizon)]

    ml = pd.read_csv(
        ML_OOS_DIR / f"{best_model}_oos_predictions_h{horizon}.csv",
        parse_dates=["date"]
    ).dropna(subset=["y_true", "proba"])

    garch = pd.read_csv(
        GARCH_OOS_DIR / f"garch_oos_predictions_h{horizon}.csv",
        parse_dates=["date"]
    ).dropna(subset=["y_true", "proba"])

    # Actual stress periods
    stress_dates = ml[ml["y_true"] == 1]["date"]
    for d in stress_dates:
        ax.axvspan(d, d + pd.Timedelta(days=1),
                   color=ACCENT_RED, alpha=0.12, zorder=1)

    # Threshold shading
    ax.axhspan(thresh, 1.0, color=ACCENT_RED, alpha=0.03, zorder=1)
    ax.axhspan(0.0, thresh, color=ACCENT_GRN, alpha=0.03, zorder=1)

    # Threshold line
    ax.axhline(thresh, color=ACCENT_RED, linewidth=1.0,
               linestyle=":", alpha=0.6, zorder=3, label=f"Threshold ({thresh:.2f})")

    # ML model
    ax.fill_between(ml["date"], ml["proba"],
                    alpha=0.10, color=ACCENT_BLUE, zorder=2)
    ax.plot(ml["date"], ml["proba"],
            label=best_label, color=ACCENT_BLUE,
            linewidth=1.8, zorder=4)

    # GARCH
    ax.fill_between(garch["date"], garch["proba"],
                    alpha=0.08, color=ACCENT_AMB, zorder=2)
    ax.plot(garch["date"], garch["proba"],
            label="GARCH", color=ACCENT_AMB,
            linewidth=1.8, linestyle="--", zorder=4)

    # Legend
    stress_patch = mpatches.Patch(
        facecolor=ACCENT_RED, alpha=0.35,
        edgecolor=ACCENT_RED, linewidth=0.8,
        label="Actual Stress Period")
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles + [stress_patch], labels + ["Actual Stress Period"],
              loc="upper left", framealpha=0.95,
              edgecolor=BORDER, fontsize=9)

    ax.set_ylabel("Predicted Probability", labelpad=7)
    ax.set_title(f"{horizon}-Day Horizon - {best_label} vs GARCH")
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
    _date_axis(ax)
    _spine(ax)

axes[-1].set_xlabel("Date", labelpad=7)

fig.suptitle("Stress Prediction Timeline - Best ML Model vs GARCH",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)

plt.savefig(PLOT_DIR / "stress_prediction_timeline.png", bbox_inches="tight")
plt.show()
print("Saved → artifacts/plots/stress_prediction_timeline.png")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

ML_OOS_DIR    = Path("../artifacts/ml_models_oos")
GARCH_OOS_DIR = Path("../artifacts/garch_oos")

MODELS = ["catboost", "xgboost", "logistic_regression"]
HORIZONS = [1, 3]


def load_proba(model, horizon):
    if model == "garch":
        path = GARCH_OOS_DIR / f"garch_oos_predictions_h{horizon}.csv"
    else:
        path = ML_OOS_DIR / f"{model}_oos_predictions_h{horizon}.csv"
    df = pd.read_csv(path, parse_dates=["date"])
    if "stress" in df.columns and "y_true" not in df.columns:
        df = df.rename(columns={"stress": "y_true"})
    return df[["date", "y_true", "proba"]].dropna()


def brier_losses(y_true, proba):
    """Per-observation squared error."""
    return (y_true - np.clip(proba, 0, 1)) ** 2


def diebold_mariano(loss1, loss2):
    """Two-sided DM test. H0: equal predictive accuracy."""
    d = loss1 - loss2
    n = len(d)
    mean_d = d.mean()
    # Newey-West variance (lag = 1)
    gamma0 = np.var(d, ddof=1)
    gamma1 = np.cov(d[:-1], d[1:])[0, 1]
    nw_var = (gamma0 + 2 * gamma1) / n
    if nw_var <= 0:
        return np.nan, np.nan
    dm = mean_d / np.sqrt(nw_var)
    p  = 2 * (1 - stats.norm.cdf(abs(dm)))
    return round(dm, 3), round(p, 4)


results = []

for horizon in HORIZONS:
    garch_df = load_proba("garch", horizon)

    for model in MODELS:
        ml_df = load_proba(model, horizon)

        merged = pd.merge(
            ml_df.rename(columns={"proba": "ml_proba"}),
            garch_df[["date", "proba"]].rename(columns={"proba": "garch_proba"}),
            on="date", how="inner"
        ).dropna()

        y           = merged["y_true"].values
        ml_loss     = brier_losses(y, merged["ml_proba"].values)
        garch_loss  = brier_losses(y, merged["garch_proba"].values)

        dm, p = diebold_mariano(ml_loss, garch_loss)

        results.append({
            "horizon":              horizon,
            "ml_model":             model,
            "brier_ml":             round(ml_loss.mean(), 4),
            "brier_garch":          round(garch_loss.mean(), 4),
            "DM_stat":              dm,
            "p_value":              p,
            "significant (p<0.05)": p < 0.05 if p is not np.nan else False,
        })

print(pd.DataFrame(results).to_string(index=False))